# SYNERA Replication (06)

In [ ]:

import pandas as pd
import os
from synera_core.discovery.interactions import factorial_interaction
from synera_core.validation.multitest import bh_fdr
from synera_core.validation.replication import replicate
from synera_core.provenance.ledger import Ledger
from synera_core.provenance.manifest import register_file
from synera_core.config import find_repo_root

repo_root = find_repo_root()
db_path = repo_root / "artifacts" / "synera.db"
key_path = repo_root / "configs" / "signer.key"
ledger = Ledger(db_path, key_path)

# Discovery (OSD-211)
expr_d = pd.read_parquet('../data/processed/osdr_OSD-211_expr.parquet')
meta_d = pd.read_parquet('../data/processed/osdr_OSD-211_meta.parquet')
meta_d = meta_d.set_index('sample_id').loc[expr_d.columns].reset_index()
res_d = factorial_interaction(expr_d.T, meta_d, 'radiation', 'microgravity')
res_d['q_AB_d'] = bh_fdr(res_d['p_AB'].values)
res_d = res_d.rename(columns={c: c+"_d" for c in res_d.columns if c != "q_AB_d"})

# Replication (OSD-237)
expr_r = pd.read_parquet('../data/processed/osdr_OSD-237_expr.parquet')
meta_r = pd.read_parquet('../data/processed/osdr_OSD-237_meta.parquet')
meta_r = meta_r.set_index('sample_id').loc[expr_r.columns].reset_index()
res_r = factorial_interaction(expr_r.T, meta_r, 'radiation', 'microgravity')
res_r = res_r.rename(columns={c: c+"_r" for c in res_r.columns})

# Replicate
rep_res = replicate(res_d, res_r, evidence_level="CROSS_TISSUE_SAME_COHORT")
print(rep_res)

# Save and register
out_path = repo_root / "artifacts" / "results"
out_path.mkdir(parents=True, exist_ok=True)
pd.DataFrame([rep_res]).drop(columns=['flags']).to_json(out_path / "track_a_replication.json")
register_file(ledger, out_path / "track_a_replication.json", "finding_verification", {"track": "A"})
    